In [17]:
!pip install torch==2.6.0 pytorch-lightning==2.0.2 pytorch-forecasting==1.4.0 scikit-learn pandas matplotlib


In [ ]:
import torch
print(torch.__file__)
print(torch.__version__)


c:\Users\clouduser\condaconda\envs\torch_eeeee\lib\site-packages\torch\__init__.py
2.1.0+cu118


In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A40-24Q


In [2]:
import pytorch_lightning as pl
import pytorch_forecasting as pf
print("Lightning:", pl.__version__)
print("Forecasting:", pf.__version__)

c:\Users\clouduser\condaconda\envs\torch_eeeee\lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Lightning: 2.1.0
Forecasting: 1.4.0


In [5]:
import pandas as pd

df = pd.read_csv(r"C:\Users\clouduser\Downloads\household_power_consumption.txt", sep=';', low_memory=False)

# Convert Date + Time to a full datetime (optional for time index)
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True, errors="coerce")
df["hour"] = df["datetime"].dt.hour
df["minute"] = df["datetime"].dt.minute
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month
df["day"] = df["datetime"].dt.day



columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]
df[columns] = df[columns].apply(pd.to_numeric, errors="coerce")

# Drop missing values
df = df.dropna().sort_values("datetime").reset_index(drop=True)

############################################################
# If you want to reduce the data size, uncomment this line #
############################################################
# df = df.sample(10000)

# Add time_idx and group
df["time_idx"] = range(len(df))
df["group"] = 0  # single time series

In [6]:
# Create a time index (sequential integer)
df['time_idx'] = range(len(df))

# Single time series => assign group id 0
df['group'] = 0


df["y"] = df["Voltage"]
# Drop rows with missing target values
df = df.dropna(subset=['y'])

# Set cutoff date for training/validation split (adjust date as needed)
training_cutoff = pd.Timestamp("2006-12-25 00:00:00")  # example cutoff

# Split data
training_data = df[df['datetime'] <= training_cutoff]
validation_data = df[df['datetime'] > training_cutoff]

In [6]:
!pip install tensorboard


   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ---------------------------------------  5.5/5.5 MB 118.3 MB/s eta 0:00:01
   ---------------------------------------- 5.5/5.5 MB 30.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 4.3/4.3 MB 44.7 MB/s eta 0:00:00

   ---------- ----------------------------- 1/4 [grpcio]
   ---------- ----------------------------- 1/4 [grpcio]
   -------------------- ------------------- 2/4 [absl-py]
   ------------------------------ --------- 3/4 [tensorboard]
   ------------------------------ --------- 3/4 [tensorboard]
   ------------------------------ --------- 3/4 [tensorboard]
   ------------------------------ --------- 3/4 [tensorboard]
   ------------------------------ --------- 3/4 [tensorboard]
   ------------------------------ --------- 3/4 [tensorboard]
   ------------------------------ --------- 3/4 [tensorboard]
   --------------------

In [8]:
import lightning.pytorch as pl
import numpy
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.tuner import Tuner
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, QuantileLoss
from pytorch_forecasting.metrics import SMAPE
logger = TensorBoardLogger("lightning_logs", default_hp_metric=False)
df = df.sort_values('datetime').reset_index(drop=True)

# Create a time index (sequential integer)
df['time_idx'] = range(len(df))

# Single time series => assign group id 0
df['group'] = 0


df["y"] = df["Voltage"]
# Drop rows with missing target values
df = df.dropna(subset=['y'])



# Set cutoff date for training/validation split (adjust date as needed)
training_cutoff = pd.Timestamp("2007-01-01 00:00:00")  # example cutoff

# Split data
training_data = df[df['datetime'] <= training_cutoff]
validation_data = df[df['datetime'] > training_cutoff]

# Parameters for the dataset
max_encoder_length = 720  # e.g., past 7 days hourly if hourly data
max_prediction_length = 	1440  # e.g., forecast next 24 hours

# Define TimeSeriesDataSet for training


training = TimeSeriesDataSet(
    training_data,
    time_idx="time_idx",
    target="y",
    group_ids=["group"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=[],
    static_reals=[],
    time_varying_known_categoricals=[],
    time_varying_known_reals=["time_idx", "hour", "day_of_week", "minute"],  # time_idx as known real covariate
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=[
      "y",
      "Global_active_power",
      "Global_reactive_power",
      "Global_intensity",
      "Sub_metering_1",
      "Sub_metering_2",
      "Sub_metering_3"
    ],
  # target is unknown in future
    allow_missing_timesteps=True,
)

# Validation dataset from training dataset to apply same normalization
validation = TimeSeriesDataSet.from_dataset(
    training,
    validation_data,
    stop_randomization=True
)

# DataLoaders
batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=2 , persistent_workers = True)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=2, persistent_workers = True)

# Callbacks and trainer setup
early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=3, verbose=True, mode="min")
lr_logger = LearningRateMonitor()
!nvidia-smi

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="gpu",
    gradient_clip_val=0.1,
    callbacks=[lr_logger, early_stop_callback],
    logger=logger,
    limit_train_batches=1.0,  # optional, speed up training for testing
    limit_val_batches=1.0,
)

# Define model
tft = TemporalFusionTransformer.from_dataset(
    training,
    hidden_size=16,
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss(),
    log_interval=10,
    learning_rate=0.03,
    reduce_on_plateau_patience=4,
)

print(f"Number of parameters in network: {tft.size()/1e3:.1f}k")

# Find learning rate
tuner = Tuner(trainer)
lr_find = tuner.lr_find(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader, max_lr=0.3)
print(f"Suggested learning rate: {lr_find.suggestion()}")

# Optionally, update learning rate
tft.hparams.learning_rate = lr_find.suggestion()

# Train
trainer.fit(tft, train_dataloader, val_dataloader)



💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
c:\Users\clouduser\condaconda\envs\torch_eeeee\lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\clouduser\condaconda\envs\torch_eeeee\lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. I

Thu Jul 24 22:55:11 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 474.04       Driver Version: 474.04       CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A40-24Q     WDDM  | 00000000:00:13.0  On |                  N/A |
| N/A    0C    P8    N/A /  N/A |   3169MiB / 24576MiB |      7%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

`Trainer.fit` stopped: `max_steps=100` reached.
Finding best initial lr: 100%|██████████| 100/100 [01:31<00:00,  1.09it/s]
Learning rate set to 0.009587315155141834
Restoring states from the checkpoint path at c:\Users\clouduser\Downloads\.lr_find_d5833bf9-56a0-4932-8ed6-1670fd922171.ckpt
Restored all states from the checkpoint at c:\Users\clouduser\Downloads\.lr_find_d5833bf9-56a0-4932-8ed6-1670fd922171.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 0      | train
3  | prescalers                         | ModuleDict                      | 176    

Suggested learning rate: 0.009587315155141834
Epoch 0: 100%|██████████| 309/309 [4:02:37<00:00,  0.02it/s, v_num=3, train_loss_step=0.801, val_loss=1.790, train_loss_epoch=1.060]

Metric val_loss improved. New best score: 1.788


...

Epoch 3: 100%|██████████| 309/309 [4:02:24<00:00,  0.02it/s, v_num=3, train_loss_step=0.589, val_loss=1.570, train_loss_epoch=0.616]

Metric val_loss improved by 0.130 >= min_delta = 0.0001. New best score: 1.571


Epoch 6: 100%|██████████| 309/309 [3:59:17<00:00,  0.02it/s, v_num=3, train_loss_step=0.530, val_loss=1.610, train_loss_epoch=0.534]

Monitored metric val_loss did not improve in the last 3 records. Best score: 1.571. Signaling Trainer to stop.


Epoch 6: 100%|██████████| 309/309 [3:59:17<00:00,  0.02it/s, v_num=3, train_loss_step=0.530, val_loss=1.610, train_loss_epoch=0.534]


In [13]:
import pandas as pd
import torch
print("\n--- Starting Prediction and Output Extraction ---")

# Make predictions on the validation dataloader to get raw outputs
# 'mode="raw"' gives you a dictionary with 'prediction' and 'attention' tensors.
# 'return_x=True' gives you the input features, which might be useful for context.
import torch
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

# Set model to eval mode
tft.eval()

# Fetch a single batch (x, y)
x, y = next(iter(val_dataloader))

# Move batch to same device as model
device = tft.device
for key in x:
    if isinstance(x[key], torch.Tensor):
        x[key] = x[key].to(device)

# Run forward pass manually
with torch.no_grad():
    out = tft.forward(x)

# Now process the output
raw_predictions = out  # contains prediction and optionally attention


print("\n--- TFT Prediction Output Structure (raw_predictions) ---")
print(f"Type of raw_predictions object: {type(raw_predictions)}")


print("Keys in raw_predictions dictionary:")
for key, value in raw_predictions.items():
    print(f"- '{key}': type={type(value)}, shape={value.shape if hasattr(value, 'shape') else 'N/A'}")

# --- Process and Save Predictions (Median and Quantiles) ---
if 'prediction' in raw_predictions.keys():
    print("\nSample of 'prediction' (first batch, first sample, all horizons, all quantiles):")
    # The 'prediction' tensor has shape (batch_size, max_prediction_length, num_quantiles)
    # For QuantileLoss (default 7 quantiles):
    # Index 0: 0.025 quantile (lower bound 95% CI)
    # Index 1: 0.1 quantile (lower bound 90% CI)
    # Index 4: 0.5 quantile (median forecast)
    # Index 5: 0.9 quantile (upper bound 90% CI)
    # Index 6: 0.975 quantile (upper bound 95% CI)

    all_forecasts = raw_predictions['prediction'].cpu().numpy()

    # For demonstration, we'll process and save the first sample's forecast.
    # If you have multiple series or batches, you might need to iterate and save multiple files.
    first_sample_forecast = all_forecasts[0] # Shape: (max_prediction_length, num_quantiles)

    # Create a DataFrame for this sample's forecast
    first_sample_predictions_df = pd.DataFrame({
        'prediction_horizon': range(1, max_prediction_length + 1), # Horizon from 1 to max_prediction_length
        'median_forecast': first_sample_forecast[:, 4],
        'lower_bound_90': first_sample_forecast[:, 1],
        'upper_bound_90': first_sample_forecast[:, 5],
        'lower_bound_95': first_sample_forecast[:, 0],
        'upper_bound_95': first_sample_forecast[:, 6],
    })
    print("\n--- Sample of Processed Forecasts (First Batch, First Sample) ---")
    display(first_sample_predictions_df.head())

    # Save the predictions to a CSV file
    first_sample_predictions_df.to_csv(r"C:\Users\clouduser\Downloads\tft_median_and_quantiles_forecast_sample_0.csv", index=False, sep=";")
    print("\nSaved median and quantile forecasts to tft_median_and_quantiles_forecast_sample_0.csv")

else:
    print(" 'prediction' key not found in raw_predictions dictionary.")


# --- Process and Save Attention Weights ---
if 'encoder_attention' in raw_predictions.keys():
    print("\nSample of 'attention' (first batch, all prediction horizons, all encoder steps):")
    # The 'attention' tensor typically has shape (batch_size, num_heads, prediction_length, encoder_length)
    # Since your attention_head_size=1, it's likely (batch_size, prediction_length, encoder_length)

    all_attention_weights = raw_predictions['encoder_attention'].cpu().numpy()

    # For simplicity, let's take the attention weights for the first sample (batch 0)
    # The shape will be (max_prediction_length, max_encoder_length)
    sample_attention = all_attention_weights[0]

    # Create DataFrame for attention weights
    # Rows are prediction horizons, columns are encoder time steps (from oldest to most recent historical)
    attention_df = pd.DataFrame(
        sample_attention[:,0,:],
        columns=[f"encoder_step_{-(max_encoder_length - i)}" for i in range(max_encoder_length)],
        index=[f"prediction_horizon_{i+1}" for i in range(max_prediction_length)]
    )
    print("\n--- Sample of Attention Weights (First Batch, All Prediction Horizons) ---")
    display(attention_df.head())

    # Save attention weights to CSV, including the index
    attention_df.to_csv(r"C:\Users\clouduser\Downloads\tft_attention_weights_sample_0.csv", index=True, sep=";")
    print("\nSaved attention weights to tft_attention_weights_sample_0.csv")

else:
    print(" 'attention' key not found in raw_predictions dictionary.")

print("--- Output Extraction Complete ---")

# After running this, you should find two new CSV files in your Jupyter directory:
# 1. tft_median_and_quantiles_forecast_sample_0.csv
# 2. tft_attention_weights_sample_0.csv

# Please download these files from your Jupyter environment and share them.
# Once I have these numerical outputs, I can then proceed with analyzing the
# LSTPrompt library (AdityaLab/lstprompt) and guide you on how to format
# this data for your LLM integration.


--- Starting Prediction and Output Extraction ---

--- TFT Prediction Output Structure (raw_predictions) ---
Type of raw_predictions object: <class 'pytorch_forecasting.utils._utils.TupleOutputMixIn.to_network_output.<locals>.Output'>
Keys in raw_predictions dictionary:
- 'prediction': type=<class 'torch.Tensor'>, shape=torch.Size([64, 1440, 7])
- 'encoder_attention': type=<class 'torch.Tensor'>, shape=torch.Size([64, 1440, 1, 720])
- 'decoder_attention': type=<class 'torch.Tensor'>, shape=torch.Size([64, 1440, 1, 1440])
- 'static_variables': type=<class 'torch.Tensor'>, shape=torch.Size([64, 0])
- 'encoder_variables': type=<class 'torch.Tensor'>, shape=torch.Size([64, 720, 1, 11])
- 'decoder_variables': type=<class 'torch.Tensor'>, shape=torch.Size([64, 1440, 1, 4])
- 'decoder_lengths': type=<class 'torch.Tensor'>, shape=torch.Size([64])
- 'encoder_lengths': type=<class 'torch.Tensor'>, shape=torch.Size([64])

Sample of 'prediction' (first batch, first sample, all horizons, all quant

,prediction_horizon,median_forecast,lower_bound_90,upper_bound_90,lower_bound_95,upper_bound_95
0,1,240.752609,239.489700,241.135712,238.817108,241.683624
1,2,240.659058,239.407501,241.027191,238.716049,241.558228
2,3,240.658340,239.408615,241.024185,238.712402,241.552536
3,4,240.633301,239.379364,241.000610,238.681152,241.530380
4,5,240.605865,239.341934,240.976852,238.639984,241.510712



Saved median and quantile forecasts to tft_median_and_quantiles_forecast_sample_0.csv

Sample of 'attention' (first batch, all prediction horizons, all encoder steps):

--- Sample of Attention Weights (First Batch, All Prediction Horizons) ---


,encoder_step_-720,encoder_step_-719,encoder_step_-718,encoder_step_-717,encoder_step_-716,encoder_step_-715,encoder_step_-714,encoder_step_-713,encoder_step_-712,encoder_step_-711,...,encoder_step_-10,encoder_step_-9,encoder_step_-8,encoder_step_-7,encoder_step_-6,encoder_step_-5,encoder_step_-4,encoder_step_-3,encoder_step_-2,encoder_step_-1
prediction_horizon_1,0.001144,0.001078,0.001096,0.001123,0.001112,0.001059,0.000485,0.000521,0.000533,0.000539,...,0.002611,0.002667,0.002699,0.001094,0.001104,0.001134,0.001147,0.001169,0.001182,0.001000
prediction_horizon_2,0.001018,0.000970,0.000994,0.001020,0.001012,0.000968,0.000506,0.000541,0.000551,0.000555,...,0.002448,0.002509,0.002541,0.001103,0.001113,0.001123,0.001149,0.001172,0.001178,0.000901
prediction_horizon_3,0.000990,0.000944,0.000970,0.000996,0.000989,0.000947,0.000507,0.000542,0.000552,0.000555,...,0.002413,0.002475,0.002508,0.001111,0.001122,0.001133,0.001159,0.001180,0.001181,0.000888
prediction_horizon_4,0.001004,0.000958,0.000983,0.001009,0.001001,0.000958,0.000512,0.000547,0.000558,0.000562,...,0.002361,0.002424,0.002457,0.001083,0.001094,0.001107,0.001130,0.001152,0.001152,0.000862
prediction_horizon_5,0.001033,0.000986,0.001007,0.001032,0.001023,0.000977,0.000516,0.000553,0.000565,0.000569,...,0.002315,0.002378,0.002410,0.001045,0.001056,0.001071,0.001092,0.001113,0.001115,0.000835



Saved attention weights to tft_attention_weights_sample_0.csv
--- Output Extraction Complete ---
